# 03.1 Statements and Expressions

Every line of Python you will ever write is one of two things: an **expression**
or a **statement**. Getting this distinction clear now explains a surprising
amount later — why some things can go inside an f-string and others cannot, why
`x = y = 0` works but `x = (y = 0)` does not, and why the REPL prints some lines
and stays silent on others.

## Theory

### An expression produces a value

An expression is anything Python can **evaluate down to a single value**.

```
2 + 3           ->  5
len("hello")    ->  5
x > 10          ->  True
[1, 2, 3]       ->  [1, 2, 3]
```

The test is simple: *could I put this on the right-hand side of an `=`?* If yes,
it is an expression.

```python
result = 2 + 3          # fine, 2 + 3 is an expression
result = len("hello")   # fine
```

### A statement performs an action

A statement is an **instruction that does something**. It does not evaluate to a
value, so you cannot assign it or nest it inside another expression.

```python
x = 5                   # assignment statement
import math             # import statement
if x > 3: ...           # conditional statement
return value            # return statement
```

Try to use one as a value and Python refuses:

```python
result = (x = 5)        # SyntaxError
```

### Why the distinction is real, not academic

Python's grammar is built on it. Wherever the language says "an expression goes
here", *only* expressions fit. That single rule explains:

- Why you can write `f"{a + b}"` but not `f"{x = 5}"` (in older Python)
- Why `[f(x) for x in items]` works but `[import x for x in items]` is nonsense
- Why the walrus operator `:=` had to be *invented* — assignment was a statement,
  and people wanted it in places where only expressions were allowed

### The four things every expression can be made of

1. **Literals** — values written directly: `42`, `"text"`, `[1, 2]`
2. **Names** — references to objects: `total`, `user_name`
3. **Operators** — combine values: `+`, `>`, `and`, `in`
4. **Calls** — invoking something: `len(x)`, `obj.method()`

Everything else is built from these four.

In [ ]:
# Expressions: each of these evaluates down to a single value.
# We assign each one, which is only possible because they ARE expressions.

literal_expression = 42
name_expression = literal_expression
operator_expression = 2 + 3
call_expression = len("hello")
comparison_expression = 10 > 3
logical_expression = True and False
collection_expression = [1, 2, 3]
method_expression = "hello".upper()

# Print each with its value and type, so the pattern is visible.
examples = [
    ("literal", literal_expression),
    ("name", name_expression),
    ("operator", operator_expression),
    ("call", call_expression),
    ("comparison", comparison_expression),
    ("logical", logical_expression),
    ("collection", collection_expression),
    ("method call", method_expression),
]

print("Kind           Value            Type")
print("-" * 46)
for kind, value in examples:
    # repr() shows the value as you would type it in source code.
    print(kind.ljust(14), repr(value).ljust(16), type(value).__name__)

## Proving the difference: `eval` vs `exec`

Python gives us two built-ins that map exactly onto this split:

- **`eval()`** accepts an *expression* and returns its value
- **`exec()`** accepts a *statement* and returns nothing

This is the cleanest possible demonstration that the two categories are real.

In [ ]:
# eval() evaluates an EXPRESSION and hands back its value.
value = eval("2 + 3")
print("eval('2 + 3') returned:", value)

# exec() runs a STATEMENT. It always returns None, because statements
# do not produce values.
returned = exec("x = 99")
print("exec('x = 99') returned:", returned)
print("but it did have an effect - x is now:", x)

# Feeding a statement to eval() fails, because eval demands an expression.
try:
    eval("y = 5")
except SyntaxError as error:
    print("")
    print("eval('y = 5') raises SyntaxError:", error.msg)

## Expression statements: the overlap

Here is the part that trips people up. An expression **can** stand alone as a
whole line. When it does, it is called an **expression statement**.

```python
len("hello")        # valid line - but the 5 is computed and thrown away
```

Python evaluates it, gets `5`, and discards it because nothing captured it.

This is exactly why `print()` exists. In a file, a bare expression produces no
visible output — the value evaporates.

In [ ]:
# A bare expression on its own line is legal, but the value is discarded.
2 + 3                    # computed, then thrown away - nothing appears

# The same expression, captured:
captured = 2 + 3
print("captured:", captured)

# The same expression, displayed:
print(2 + 3)

# This is why calling a method and ignoring the result is a common bug:
text = "  hello  "

# strip() RETURNS a new string - it does not modify text in place.
text.strip()             # the stripped value is computed and discarded
print("after bare text.strip():", repr(text))

# You must capture the result.
text = text.strip()
print("after text = text.strip():", repr(text))

## Under the hood: what the bytecode shows

The distinction is visible in the compiled bytecode. An expression statement ends
with `POP_TOP` — the instruction that says *"throw away the value on the stack"*.

An assignment ends with `STORE_NAME` instead — *"put that value somewhere"*.

In [ ]:
import dis

print("=== A bare expression statement ===")
# The value is computed then immediately discarded.
dis.dis(compile("2 + 3", "<demo>", "exec"))

print("")
print("=== The same value, assigned ===")
# The value is computed then stored under a name.
dis.dis(compile("x = 2 + 3", "<demo>", "exec"))

Read the two listings above:

- The first ends `POP_TOP` — compute, then discard.
- The second ends `STORE_NAME` — compute, then keep.

That single instruction difference *is* the difference between an expression
statement and an assignment statement.

Notice too that both listings show `LOAD_CONST 5`, not an addition. Python did
the arithmetic at compile time — the **constant folding** you met in Chapter 01.

## The complete list of Python statements

There are not many. This is the whole vocabulary of "things that do something".

In [ ]:
# Each entry is (statement, example, what it does).
statements = [
    ("assignment", "x = 5", "bind a name to a value"),
    ("augmented assignment", "x += 1", "update in place"),
    ("annotated assignment", "x: int = 5", "bind with a type hint"),
    ("expression statement", "print(x)", "evaluate, discard the result"),
    ("if / elif / else", "if x > 0: ...", "choose a branch"),
    ("for", "for i in items: ...", "repeat per item"),
    ("while", "while x < 10: ...", "repeat while true"),
    ("break", "break", "leave the loop"),
    ("continue", "continue", "skip to the next pass"),
    ("pass", "pass", "do nothing (a placeholder)"),
    ("def", "def f(): ...", "define a function"),
    ("return", "return x", "send a value back"),
    ("yield", "yield x", "produce a value, pause"),
    ("class", "class C: ...", "define a class"),
    ("import", "import math", "load a module"),
    ("try / except / finally", "try: ...", "handle errors"),
    ("raise", "raise ValueError()", "signal an error"),
    ("with", "with open(f) as x:", "manage a resource"),
    ("assert", "assert x > 0", "check an assumption"),
    ("global", "global counter", "rebind a module-level name"),
    ("nonlocal", "nonlocal total", "rebind an enclosing name"),
    ("del", "del x", "remove a name or item"),
    ("match", "match value: ...", "structural pattern matching"),
]

print("Statement                 Example                 Does")
print("-" * 74)
for name, example, does in statements:
    print(name.ljust(25), example.ljust(23), does)

print("")
print("Total statement types:", len(statements))
print("Everything else you write is an expression.")

## Where only expressions are allowed

These positions in Python's grammar accept expressions and nothing else. Knowing
the list saves you from a whole category of `SyntaxError`.

In [ ]:
values = [1, 2, 3, 4, 5]
threshold = 3

# 1. The right-hand side of an assignment.
total = sum(values)

# 2. A function argument.
print("1. as an argument:", max(values))

# 3. Inside an f-string's braces.
print(f"2. inside an f-string: {sum(values) / len(values)}")

# 4. The condition of an if or while.
if len(values) > threshold:
    print("3. as a condition: the list is longer than", threshold)

# 5. The output part of a comprehension.
doubled = [value * 2 for value in values]
print("4. in a comprehension:", doubled)

# 6. A return value.
def average_of(numbers):
    # The whole expression is evaluated, then returned.
    return sum(numbers) / len(numbers)

print("5. as a return value:", average_of(values))

# 7. Inside a collection literal.
summary = {"count": len(values), "total": sum(values)}
print("6. in a dict literal:", summary)

# 8. As an operand of another expression.
print("7. nested in another expression:", (sum(values) * 2) - min(values))

## The walrus operator: making assignment an expression

For most of Python's life, assignment was **strictly** a statement. That caused a
recurring annoyance: you often want to compute something, keep it, *and* test it
in one step.

Python 3.8 added `:=`, the **walrus operator** — an assignment that is also an
expression, so it fits where only expressions are allowed.

```python
# Without walrus - the call happens twice, or needs an extra line.
value = expensive()
if value > 10:
    ...

# With walrus - assign and test together.
if (value := expensive()) > 10:
    ...
```

Covered fully in Chapter 06. It is introduced here because it exists *entirely*
because of the expression/statement divide.

In [ ]:
readings = [12, 45, 7, 88, 23]

# WITHOUT the walrus: you need a separate line to keep the value.
largest = max(readings)
if largest > 50:
    print("without walrus: peak is", largest)

# WITH the walrus: assign inside the condition itself.
if (peak := max(readings)) > 50:
    print("with walrus:    peak is", peak)

# It is genuinely useful in a while loop, where the value must be
# recomputed each pass AND tested.
queue = [5, 4, 3, 2, 1]

print("")
print("draining the queue:")
while queue and (item := queue.pop()) < 4:
    # item was assigned in the condition, and is usable in the body.
    print("   took", item)

print("stopped with queue:", queue)

## Common errors this explains

Each of the errors below comes from putting a statement where an expression
belongs. They look unrelated until you know the rule.

In [ ]:
# Each entry is (broken code, why it fails).
broken_examples = [
    ("x = (y = 5)", "assignment is a statement, not an expression"),
    ("[import os]", "import is a statement and cannot go in a list"),
    ("f(return 5)", "return is a statement and cannot be an argument"),
    ("x = if True: 1", "if is a statement - use a conditional expression"),
    ("[x = 1 for x in r]", "cannot assign inside a comprehension (use :=)"),
]

for source, reason in broken_examples:
    try:
        compile(source, "<demo>", "exec")
        print(source.ljust(22), "-> compiled (unexpected)")
    except SyntaxError:
        # We only print the reason, since the message wording varies by version.
        print(source.ljust(22), "-> SyntaxError:", reason)

print("")
print("The fix for the if-statement case is a conditional EXPRESSION:")

condition = True
# This is an expression, so it can go on the right of an =.
result = 1 if condition else 0
print("   result = 1 if condition else 0  ->", result)

## Takeaways

1. An **expression** evaluates to a value; a **statement** performs an action.
2. The test: *can it go on the right-hand side of an `=`?*
3. `eval()` takes expressions; `exec()` takes statements and returns `None`.
4. An expression alone on a line is an **expression statement** — its value is
   discarded, which is why `text.strip()` on its own does nothing useful.
5. In bytecode, a discarded expression ends in `POP_TOP`; an assignment ends in
   `STORE_NAME`.
6. Python has about 23 statement types. Everything else is an expression.
7. The walrus `:=` exists precisely because assignment was a statement.

## Try it yourself

1. For each of these, decide expression or statement *before* running it through
   `eval()`: `3 * 7`, `x = 1`, `"a" in "cat"`, `del x`, `[i for i in range(3)]`.
2. Run `dis.dis(compile("len('abc')", "<d>", "exec"))`. Find the `POP_TOP`.
3. Write a line that calls `.upper()` on a string and forgets to capture the
   result. Prove with `print()` that the original is unchanged.
4. Rewrite `total = max(nums); if total > 5:` using the walrus operator.